# Nonlinear static analysis

A rectangular isotropic plate, with $w$ constrained at the nodes of the four
edges, compressed along $x$ below its critical load, with a transverse force
at the centre large enough to reach a deflection of about 2.6 times the
thickness. The model keeps the value arrays of
$\boldsymbol{K}_{CNL} + \boldsymbol{K}_{GNL}$ and $\boldsymbol{K}_G$, which
are zeroed and recomputed at each iteration with `update_KCNL` and
`update_KG`, and the internal forces of `update_fint`. The code is that of
the unit test `tests/test_static_axial_load_nonlinear.py`.

The imports and the module-level definitions of the test, with the helper
functions that the code below uses:

In [1]:
import numpy as np
from numpy import isclose
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh, spsolve

from composites import isotropic_plate
from structsolve import Analysis
from structsolve.convergence import crisfield_weights, crisfield_zeta

from bfscplate2d import (BFSCPlate2D, update_KC0, update_KCNL, update_KG,
        update_fint, DOF, DOUBLE, INT, KC0_SPARSE_SIZE, KCNL_SPARSE_SIZE,
        KG_SPARSE_SIZE)
from bfscplate2d.quadrature import get_points_weights

# number of nodes
nx = 13
ny = 7

# geometry
a = 0.9
b = 0.5

# material properties
E = 0.7e11
nu = 0.3
h = 0.001


class Model(object):
    """Mesh, elements, boundary conditions and the operators built on them."""


def forces(m, load, transverse):
    """Axial compression along x, plus a transverse load at the centre."""
    fext = np.zeros(m.N)
    checkBottomEdge = isclose(m.x, 0)
    checkTopEdge = isclose(m.x, a)
    fext[0::DOF][checkBottomEdge] = +load/ny
    assert isclose(fext.sum(), load)
    fext[0::DOF][checkTopEdge] = -load/ny
    assert isclose(fext.sum(), 0)

    ftrans = np.zeros(m.N)
    check = isclose(m.x, a/2) & isclose(m.y, b/2)
    assert check.sum() == 1
    ftrans[6::DOF][check] = transverse
    return fext, ftrans


class Callables(object):
    """The callables of structsolve.Analysis for the model ``m`` under the
    external forces ``fext``.

    They loop over the elements with the update functions and assemble, and
    work on the unknown degrees of freedom only, the partition of the free
    dofs of structsolve: the solution vector of the solvers is ``c =
    u[bu]``, and the force vectors and matrices are restricted to ``bu``.
    ``calc_fint`` also keeps the load factor and the internal forces of each
    iteration, for the residual history.
    """
    def __init__(self, m, fext):
        self.m = m
        self.fext = fext
        self.iterations = []

    def expand(self, c):
        """The displacements of the model from those of the unknown dofs."""
        u = np.zeros(self.m.N)
        u[self.m.bu] = c
        return u

    def calc_fext(self, inc=1., silent=False):
        return inc*self.fext[self.m.bu]

    def calc_fint(self, c, inc=1., silent=False):
        fint = calc_fint(self.m, self.expand(c))[self.m.bu]
        self.iterations.append((inc, fint))
        return fint

    def calc_kC(self, c=None, NLgeom=False, silent=False):
        if not NLgeom:
            return self.m.KC0uu
        bu = self.m.bu
        KCNL = calc_KCNL(self.m, self.expand(c))
        return (self.m.KC0 + KCNL)[bu, :][:, bu]

    def calc_kG(self, c=None, NLgeom=False, silent=False):
        bu = self.m.bu
        return calc_KG(self.m, self.expand(c))[bu, :][:, bu]


def newton_raphson(m, fext, increments=1):
    """Full Newton-Raphson of structsolve.Analysis, with ``increments``
    equal load increments. Returns the solution and the residual history of
    the last increment.

    The tangent is rebuilt every iteration, so the residual history is the
    direct evidence of whether KT is the derivative of fint: a consistent
    tangent squares the residual each step, an inconsistent one multiplies
    it by a constant factor instead. The residual is measured with
    Crisfield's criterion, the convergence criterion of structsolve, with
    its default tolerance of 1e-9.
    """
    callables = Callables(m, fext)
    an = Analysis(calc_fext=callables.calc_fext,
                  calc_fint=callables.calc_fint,
                  calc_kC=callables.calc_kC,
                  calc_kG=callables.calc_kG)
    an.initialInc = 1./increments
    an.maxInc = 1./increments
    increments_done, cs = an.static(NLgeom=True)
    # every increment converged, none was cut
    assert np.allclose(increments_done,
                       np.linspace(0, 1, increments + 1)[1:]), (
            'Not converged! load factors %s' % increments_done)
    u = callables.expand(cs[-1])

    w = crisfield_weights(m.KC0uu)
    f = fext[m.bu]
    history = [crisfield_zeta(inc*f - fint, inc*f, fint, w)
               for inc, fint in callables.iterations if inc == 1.]
    return u, history


def assert_quadratic_convergence(history, epsilon=1.e-9):
    """The last Newton step must square the residual, not merely shrink it.

    Quadratic convergence is an asymptotic statement, so it is the final
    transition that carries it: the step that takes the iterate from above
    the tolerance to below it. Earlier steps are not a fair test here,
    because the residual is normalised by the whole external load, which the
    axial part may satisfy at the first step while the out-of-plane part is
    still far from converged.

    With an inconsistent tangent the residual falls by a roughly constant
    factor throughout, so no step ever squares it and this fails.
    """
    assert history[-1] < epsilon, 'did not reach the tolerance'
    assert len(history) <= 6, (
        'took %d iterations to converge; a consistent tangent needs a '
        'handful' % len(history))
    assert len(history) >= 2, 'no Newton step was taken'
    prev, curr = history[-2], history[-1]
    # the floor keeps the test meaningful once prev**2 drops below the
    # round-off level of the residual itself, about 1e-14 here: the last
    # increment of test_large_deflection goes 2.7e-8 -> 1.5e-14
    assert curr < max(10.*prev**2, 1.e-13), (
        'the last step took the residual %.3e -> %.3e, which is linear '
        'rather than quadratic; the tangent is not the derivative of fint'
        % (prev, curr))


def linear_buckling_load(m, load):
    """The first buckling load, for the reference amplification."""
    fext, _ = forces(m, load, 0.)
    u0 = np.zeros(m.N)
    u0[m.bu] = spsolve(m.KC0uu, fext[m.bu])
    KGuu = calc_KG(m, u0)[m.bu, :][:, m.bu]
    eigvals, eigvecsu = eigsh(A=KGuu, k=5, which='SM', M=m.KC0uu, tol=1e-6,
                              sigma=1., mode='cayley')
    eigvals = -1./eigvals
    return eigvals[0]*load, u0

In [2]:
def build_model():
    points, weights = get_points_weights(nint=4)
    lam = isotropic_plate(thickness=h, E=E, nu=nu)

    xlin = np.linspace(0, a, nx)
    ylin = np.linspace(0, b, ny)
    xmesh, ymesh = np.meshgrid(xlin, ylin)

    # getting nodes
    ncoords = np.vstack((xmesh.T.flatten(), ymesh.T.flatten())).T
    x = ncoords[:, 0]
    y = ncoords[:, 1]
    nids = 1 + np.arange(ncoords.shape[0])
    nid_pos = dict(zip(nids, np.arange(len(nids))))

    nids_mesh = nids.reshape(nx, ny)

    n1s = nids_mesh[:-1, :-1].flatten()
    n2s = nids_mesh[1:, :-1].flatten()
    n3s = nids_mesh[1:, 1:].flatten()
    n4s = nids_mesh[:-1, 1:].flatten()

    num_elements = len(n1s)
    print('num_elements', num_elements)

    N = DOF*nx*ny
    KC0r = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    KC0c = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
    KC0v = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    KCNLr = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
    KCNLc = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=INT)
    KCNLv = np.zeros(KCNL_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    KGr = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
    KGc = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=INT)
    KGv = np.zeros(KG_SPARSE_SIZE*num_elements, dtype=DOUBLE)
    init_k_KC0 = 0
    init_k_KCNL = 0
    init_k_KG = 0

    elements = []
    for n1, n2, n3, n4 in zip(n1s, n2s, n3s, n4s):
        plate = BFSCPlate2D()
        plate.n1 = n1
        plate.n2 = n2
        plate.n3 = n3
        plate.n4 = n4
        plate.c1 = DOF*nid_pos[n1]
        plate.c2 = DOF*nid_pos[n2]
        plate.c3 = DOF*nid_pos[n3]
        plate.c4 = DOF*nid_pos[n4]
        plate.ABD = lam.ABD
        plate.lex = a/(nx - 1)
        plate.ley = b/(ny - 1)
        plate.init_k_KC0 = init_k_KC0
        plate.init_k_KCNL = init_k_KCNL
        plate.init_k_KG = init_k_KG
        update_KC0(plate, points, weights, KC0r, KC0c, KC0v)
        init_k_KC0 += KC0_SPARSE_SIZE
        init_k_KCNL += KCNL_SPARSE_SIZE
        init_k_KG += KG_SPARSE_SIZE
        elements.append(plate)

    KC0 = coo_matrix((KC0v, (KC0r, KC0c)), shape=(N, N)).tocsc()

    # applying boundary conditions
    # simply supported
    bk = np.zeros(N, dtype=bool)
    checkSS = isclose(x, 0) | isclose(x, a)
    bk[3::DOF] = checkSS
    bk[6::DOF] = checkSS
    checkSS = isclose(y, 0) | isclose(y, b)
    bk[6::DOF] += checkSS
    check = isclose(x, a/2.)
    bk[0::DOF] = check
    bu = ~bk
    print('boundary conditions bu', bu.sum())

    m = Model()
    m.points, m.weights, m.lam = points, weights, lam
    m.ncoords, m.x, m.y = ncoords, x, y
    m.xmesh, m.ymesh = xmesh, ymesh
    m.nid_pos = nid_pos
    m.elements = elements
    m.N = N
    m.KC0 = KC0
    m.KC0uu = KC0[bu, :][:, bu]
    m.bk, m.bu = bk, bu
    m.KCNLr, m.KCNLc, m.KCNLv = KCNLr, KCNLc, KCNLv
    m.KGr, m.KGc, m.KGv = KGr, KGc, KGv
    return m

In [3]:
def calc_KCNL(m, u):
    """The nonlinear part KCNL(u) of the tangent, K_CNL + K_GNL."""
    m.KCNLv *= 0
    for plate in m.elements:
        update_KCNL(u, plate, m.points, m.weights, m.KCNLr, m.KCNLc, m.KCNLv)
    return coo_matrix((m.KCNLv, (m.KCNLr, m.KCNLc)),
                      shape=(m.N, m.N)).tocsc()

In [4]:
def calc_KG(m, u):
    """The geometric stiffness KG(u), of the tangent and of the linearised
    comparisons."""
    m.KGv *= 0
    for plate in m.elements:
        update_KG(u, plate, m.points, m.weights, m.KGr, m.KGc, m.KGv)
    return coo_matrix((m.KGv, (m.KGr, m.KGc)), shape=(m.N, m.N)).tocsc()

In [5]:
def calc_fint(m, u):
    fint = np.zeros(m.N)
    for plate in m.elements:
        update_fint(u, plate, m.points, m.weights, fint)
    return fint

The full Newton-Raphson method of structsolve, with the tangent stiffness
matrix $\boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} + \boldsymbol{K}_{GNL} +
\boldsymbol{K}_G$ rebuilt at every iteration and the external forces applied
in four increments, `newton_raphson(m, fext + ftrans, increments=4)`, see the
notebook *Nonlinear static analysis with structsolve*. The test checks the
quadratic convergence and that the membrane stretching makes the plate
stiffer than the linearised prediction
$(\boldsymbol{K}_{C0} + \boldsymbol{K}_G)^{-1}\boldsymbol{F}_{ext}$, with
$\boldsymbol{K}_G$ of the axial load:

In [6]:
def test_large_deflection(plot=False):
    """Transverse load reaching w/h of about 2.6: membrane stiffening."""
    load = 300.  # N
    transverse = 5.  # N

    m = build_model()
    fext, ftrans = forces(m, load, transverse)

    load_cr, u_axial = linear_buckling_load(m, load)

    # the linearised answer the plate would give if the von Karman terms did
    # not stiffen it as it deflects
    KG = calc_KG(m, u_axial)
    u_lin = np.zeros(m.N)
    u_lin[m.bu] = spsolve((m.KC0 + KG)[m.bu, :][:, m.bu], ftrans[m.bu])
    w_lin = u_lin[6::DOF].max()

    u, history = newton_raphson(m, fext + ftrans, increments=4)
    assert_quadratic_convergence(history)

    w = u[6::DOF].reshape(nx, ny).T
    print('w min max', w.min(), w.max())
    print('w/h', w.max()/h)
    print('w linearised', w_lin)

    if plot:
        import matplotlib
        matplotlib.use('TkAgg')
        import matplotlib.pyplot as plt
        plt.gca().set_aspect('equal')
        levels = np.linspace(w.min(), w.max(), 300)
        plt.contourf(m.xmesh, m.ymesh, w, levels=levels)
        plt.colorbar()
        plt.show()

    # the regime is genuinely nonlinear
    assert 2. < w.max()/h < 3.

    # and membrane stretching makes the plate stiffer than the linearised
    # prediction, by a wide margin at this deflection
    assert w.max() < 0.6*w_lin

    assert isclose(w.max(), 2.5826494310546447e-03, rtol=1e-3)

Running the analysis:

In [7]:
test_large_deflection()

num_elements 72
boundary conditions bu 853
Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...
	Step 1, attempting load factor 0.25000
		Iteration: 1, max(|R|) = 4.913e+01, zeta = 9.276e-01


		Iteration: 2, max(|R|) = 2.479e-02, zeta = 6.075e-03


		Iteration: 3, max(|R|) = 1.780e-02, zeta = 9.610e-04


		Iteration: 4, max(|R|) = 3.212e-08, zeta = 2.977e-09


		Iteration: 5, max(|R|) = 3.821e-14, zeta = 9.778e-15
		Converged at load factor 0.25000
	Step 2, attempting load factor 0.50000
		Iteration: 1, max(|R|) = 1.059e+02, zeta = 9.343e-01


		Iteration: 2, max(|R|) = 7.471e-01, zeta = 2.605e-02


... (7 outputs left out)

		Iteration: 2, max(|R|) = 2.525e-01, zeta = 6.541e-03


		Iteration: 3, max(|R|) = 4.185e-02, zeta = 5.367e-04


		Iteration: 4, max(|R|) = 1.646e-06, zeta = 2.709e-08


		Iteration: 5, max(|R|) = 6.083e-13, zeta = 1.484e-14
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000
w min max 0.0 0.002582649207271954
w/h 2.5826492072719542
w linearised 0.00555262647010334
